# Dunnhumby NGCF · 개인 계정 7시드 전용

이 계정은 **seed42~48만** 학습합니다. 학교 계정은50·51만 담당합니다. seed49는 기존 원본 재사용입니다. 기존 출력·checkpoint·고정 학습 source를 유지합니다.

고정 조건은 변경하지 않습니다: NGCF 8모형, 300epoch 마지막점 1회 test,
DAY1~704 학습/DAY705~711 신규상품 추천, k1, uniform K1, batch8192,
Adam .0005, M2/M3/M4 공동학습, L4. validation/holdout/조기종료/중간test 없음.
최종 계획은8모형×10시드이며 역할별 완료는 전체80개 완료와 다릅니다.

실행 중인 옛9시드 셀은 중단 후 이 버전으로 재개해야 합니다. 개인 배정 확인서를 학교 코드에 전달하기 전에는 학교 학습을 시작하지 마세요. 옛 노트북/다른 런타임의 실행을 원격으로 감시하는 기능은 없습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. 기존 조건과 같은 L4로 실행하세요.'
SOURCE_COMMIT = '63146eff4ed60bfb0cc430b77d58fdc40e4db7c5'
REPO = Path('/content/clv-dh-ngcf-remaining9-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)
import hashlib, urllib.request, importlib.util
HELPER_COMMIT = '78380e4bca3a293be28bab5336ae9b16feffb7b6'
HELPERS = {
    'clv_dh_ngcf_split72': '96c0b3d9a08d2a244c1bc15ea1967fa3414b85f46754cdb69ad9bfeba2b5fb43',
    'clv_ngcf_school_storage': '11d8b6dba321cfb72da1715208df4a45adf4495625394ff7f1d72fc25bebe0d4',
}
for name, expected in HELPERS.items():
    url = f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{HELPER_COMMIT}/{name}.py'
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected
    target = Path('/content') / (name + '.py')
    target.write_bytes(data)
    spec = importlib.util.spec_from_file_location(name, target)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import clv_dh_ngcf_split72 as split
import clv_ngcf_school_storage as school_storage
print('고정 학습코드63146eff / 분담도우미:', HELPER_COMMIT)


## 1. 기존 입력·출력·체크포인트 그대로 준비
원래 참조/출력 경로를 바꾸지 마세요. 입력 준비는 학습하지 않습니다.

In [ ]:
import clv_dh_ngcf_remaining9 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '구버전 모듈이 메모리에 남았습니다. 이 런타임만 다시 시작하세요.'
assert Path(screen.original.__file__).resolve().parent == REPO.resolve(), '기존 NGCF 모듈이 다른 경로입니다. 이 런타임만 다시 시작하세요.'
cfg = screen.configure(
    # seed49_reference='/content/drive/MyDrive/dh_ngcf_final8_seed49_results.zip',
    # m2_reference='/content/drive/MyDrive/clv_m2_adaptive_dunnhumby_final10_results.zip',
    # m4_reference='/content/drive/MyDrive/clv_m4_user_centered_final10_results.zip',
)
print('seed49 원본:', cfg.seed49_reference)
print('M2 입력 참조:', cfg.m2_reference)
print('M3/M4 입력 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
assert cfg.seeds == tuple(range(42, 52)) and cfg.epochs == 300
print('새 학습 시드:', screen.NEW_SEEDS)
print('각 시드 순서:', screen.ORDER)
print('진행·checkpoint·결과 저장:', prepared['run_dir'])
print(json.dumps(screen.report(prepared)['reading'], ensure_ascii=False, indent=2))

## 2. 7시드 배정·완료 재사용·체크포인트 재개
50·51 호출은 run_arm 수준에서도 차단합니다. 배정 확인서는 생성 직후 학교 코드에 전달할 수 있습니다. 역할별 완료56개를 전체80개 완료로 표시하지 않습니다.

In [ ]:
receipt = split.install(prepared, 'personal')
receipt_path = Path('/content/drive/MyDrive/dh_ngcf_personal7_assignment.json')
screen.base.io._atomic_json(receipt_path, receipt)
print('학교에 전달할 배정 확인서:', receipt_path)
try:
    result = split.run(prepared, 'personal')
finally:
    output = split.export_partition(prepared, 'personal',
        '/content/drive/MyDrive/dh_ngcf_personal7_results.zip')
    print('완료된 결과 패킷:', output)
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result['complete'] and result['completed'] == 56


## 3. 결과 패킷 다운로드
이 ZIP과 학교2시드 ZIP을 합산 노트북에 넣으면80개가 모두 있을 때만10시드 평균·표본 표준편차를 생성합니다. checkpoint는 원래 Drive에 유지합니다.

In [ ]:
from google.colab import files
files.download(split.export_partition(prepared, 'personal', '/content/dh_ngcf_personal7_results.zip'))
